# 04 — Condição do céu: a unidade do rótulo, capacidade e ensemble

## O que este notebook compra que a RTX 2060 não compra

O alvo primário é a **condição do céu** (as quatro classes de Escobedo, binadas em Kt), com a
difusa e o k\* ao lado. O que se sabe em 2026-09-05, medido no teste local (7.138 frames = 1.477
blocos do datalogger):

- um oráculo com o Kt **instantâneo** perfeito faz macro-F1 **0,73** contra o rótulo de média de
  5 min — o braço `ceu` (fine-tune, por frame) já faz 0,70, a 96 % desse teto;
- **por bloco do sensor** o teto sobe para 0,84 (janela centrada) e 0,84–0,99 (janela alinhada
  ao bloco) — é a única mudança que move o teto;
- a baseline honesta é a classe do **bloco anterior** (0,66 por frame / 0,67 por bloco; a
  persistência de 1 min, 0,91, é artefato — 79 % dos minutos consecutivos partilham a linha);
- o piso de decisão é **≈ 0,03** de macro-F1 (bootstrap por bloco); "+1 ponto" é ruído;
- difusa: `ceu` last.ckpt 17,6 ± 0,3 W/m² por semente, 16,7 no ensemble de 3, 16,6 heterogêneo.

Os arms, na ordem do plano (custos são estimativa até o arm S medir o tempo real):

| arm | hipótese | mecanismo | custo (H100/A100) |
|---|---|---|---|
| **S. portão** | reproduz o `ceu` local e mede o tempo/época | mesma rede, mesma receita; refuse se divergir | ~1 h |
| **U. unidade do rótulo** | `sensor_block` + uma amostra por bloco | a amostra são os 4–5 frames da mesma linha do CR5000; 60 épocas | ~2 h |
| **O. alvo ordinal** | pesos por classe + rótulo suave ordinal sobre o vencedor de S/U | ataca o recall das parciais e a CE que explode | ~2 h |
| **B. capacidade** | ViT-B/14 sobre o vencedor | só depois de U; portão de +0,03 por bloco na validação | ~3 h |
| **E. ensemble** | 5 sementes do vencedor + heterogêneo | escolhidos na **validação**, nunca no teste | ~2 h |

Cada arm é avaliado em **best e last**, no **teste e na validação**, e reportado por frame e por
bloco. Cada arm arquiva no Drive na mesma célula do treino.

## A receita comum

`CEU_TARGETS`: sky (CE) + kindex (k\*, MAE) + dhi (MAE em `DHI/DHI_céu-claro`), pesos 1/1/1;
`cls+mean`; cosseno completo (paciência = teto); `weight_decay` 0,05; jitter de exposição,
ruído e erase; sem flip/rotação. Referências no `INDICE.md` do TCC.

## Antes de rodar

1. `BRANCH` tem de existir no GitHub **com** o `_colab_runner` deste notebook (`CEU_TARGETS`,
   `ensemble_predictions`, `score_by_sensor_block`, `checkpoint=` no `run_experiment`) e com
   `alignment.strategy: sensor_block` no pacote.
2. Bundle: `bundle-iso.tar.gz` (dataset-iso com frames, 1,3 GB) em `MyDrive/labmim/allsky-mm/`.
3. `LOCAL_REFERENCE`: copie de `experiments/ceu/ensemble/metrics.json` (`by_sensor_block`).

## 1. Runtime e GPU

`Runtime > Change runtime type > A100 GPU`, *High-RAM* desligado: o pico do ViT-B com janela
de 5 frames fica em ~20 GB, longe dos 40.

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout)

## 2. Ambiente

Única célula que não pode vir do `_colab_runner`: é ela que clona o repositório onde ele
mora. O torch CUDA é instalado e **verificado** — o extra `allsky` fixa uma wheel de CPU.

In [ ]:
import os
import subprocess
import sys

REPO = "https://github.com/Bruno-Mascarenhas/micrometeorology.git"
BRANCH = "main"  # precisa carregar o _colab_runner com CEU_TARGETS e ensemble_predictions
WORKDIR = "/content/micrometeorology"

if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, WORKDIR], check=True)
subprocess.run(["pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.14"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "venv", "--python", "3.14", ".venv"], cwd=WORKDIR, check=True)
subprocess.run(["uv", "sync", "--locked", "--extra", "allsky"], cwd=WORKDIR, check=True)
subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--python",
        ".venv/bin/python",
        "--reinstall",
        "--torch-backend",
        "cu130",
        "torch==2.13.0",
    ],
    cwd=WORKDIR,
    check=True,
)

PY = f"{WORKDIR}/.venv/bin/python"
os.environ["PATH"] = f"{WORKDIR}/.venv/bin:" + os.environ["PATH"]
sys.path.insert(0, f"{WORKDIR}/notebooks/colab")

verify = subprocess.run(
    [PY, "-c", "import torch; print(torch.__version__, torch.cuda.is_available())"],
    capture_output=True,
    text=True,
    check=False,
)
print(verify.stdout)
if "True" not in verify.stdout:
    raise RuntimeError("torch sem CUDA — pare e reinstale antes de treinar")

import _colab_runner as runner  # noqa: E402

if not hasattr(runner, "ensemble_predictions") or not hasattr(runner, "CEU_TARGETS"):
    raise RuntimeError(
        f"o _colab_runner de {BRANCH} nao tem CEU_TARGETS/ensemble_predictions — "
        "aponte BRANCH para uma branch que os carregue"
    )

## 3. Dados e artefatos

`stage_bundle` copia o bundle para o SSD local, desempacota e roda `validate-dataset`.
O bundle tem de ser o do **dataset-iso** (disco concêntrico, raio 112 px em 224).

In [ ]:
import os

from google.colab import drive

BUNDLE = "/content/drive/MyDrive/labmim/allsky-mm/bundle-iso.tar.gz"
DATA = "/content/allsky-mm"
ARTIFACTS = "/content/drive/MyDrive/labmim/runs/allsky-ceu"

# Ensemble do braco `ceu` local (output/allsky-mm/experiments/ceu/ensemble/metrics.json):
# copie aqui dhi.rmse e sky.vote.balanced_accuracy quando ele terminar. None = sem portao
# local; o arm S compara so com a referencia congelada.
LOCAL_REFERENCE = {"dhi_rmse": None, "sky_balanced_accuracy": None}
FROZEN_REFERENCE_BALANCED_ACCURACY = 0.65  # f_cen_clsmean_mt, embeddings congelados

drive.mount("/content/drive")
os.makedirs(ARTIFACTS, exist_ok=True)
ROOT = runner.stage_bundle(BUNDLE, DATA, python=PY)

## 4. Hardware e ajustes que dependem dele

`bf16` existe em toda GPU do Colab menos a T4. O probe roda no interpretador do venv.

In [ ]:
import json

probe = subprocess.run(
    [
        PY,
        "-c",
        'import json, sys; sys.path.insert(0, "' + WORKDIR + '/notebooks/colab"); '
        "import _colab_runner as r; print(json.dumps(r.probe_accelerator()))",
    ],
    capture_output=True,
    text=True,
    check=True,
)
HW = json.loads(probe.stdout.strip().splitlines()[-1])
AMP_DTYPE = HW["amp_dtype"]
WORKERS = min(8, HW["cpus"])
print(HW)
print(f"amp={AMP_DTYPE}  workers={WORKERS}")

## 5. A receita comum, e o arm S — o portão

Um `arm()` só, para os cinco arms compartilharem o mesmo caminho e diferirem apenas no que
cada célula sobrescreve. `epochs` = `patience` = 40 para o cosseno anelar de verdade.

**Cada arm é avaliado em dois checkpoints.** Medido no `ceu` local em 2026-09-05: a entropia
cruzada de céu na validação sobe monotonicamente a partir da época 2 (0,61 → 1,03 na época 7)
enquanto o MAE de difusa cai (20,9 → 15,1 W/m²) — a cabeça de classe fica confiante e errada
numa val 2× mais difícil que o teste, e o `val/loss` composto trava o `best.ckpt` cedo. Por isso
`best` (a escolha do monitor) e `last` (o fim do cosseno) entram os dois na tabela, com sufixo
`_last`; qual serve a cada cabeça é um resultado, não uma premissa.

In [ ]:
from pathlib import Path

import pandas as pd

CFG = Path(WORKDIR) / "configs/allsky/experiments/colab"
OUT = Path("/content/out")
rows = []

MODEL = {
    "backbone_frozen": False,
    "unfreeze_last_n": 12,
    "image_size": 224,
    "backbone_pooling": "cls+mean",
}
TRAIN = {
    "backbone_lr": 1e-5,
    "epochs": 40,
    "batch_size": 64,
    "weight_decay": 0.05,
    "num_workers": WORKERS,
    "amp": {"enabled": True, "dtype": AMP_DTYPE},
    "early_stopping": {"patience": 40, "monitor": "val/loss"},
}
AUGMENTATION = {
    "p_exposure": 0.5,
    "exposure_log2": 0.6,
    "p_noise": 0.3,
    "noise_sigma": 0.01,
    "p_erase": 0.25,
}


def arm(name, seed, note, *, model=None, train=None, alignment=None, targets=None):
    """Run one arm, score best/last on test and best on val, archive, record its row."""
    config = runner.write_config(
        CFG / f"{name}.yaml",
        extends=["../_base.yaml", "../../models/image_only.yaml"],
        name=name,
        output_dir=str(OUT / name),
        seed=seed,
        data_root=ROOT,
        model={**MODEL, **(model or {})},
        train={**TRAIN, **(train or {})},
        targets=targets or runner.CEU_TARGETS,
        alignment=alignment,
        augmentation=AUGMENTATION,
        note=note,
    )
    row = runner.run_experiment(config, python=PY)
    last = runner.run_experiment(config, python=PY, checkpoint="last")
    val = runner.run_experiment(config, python=PY, split="val")
    for key in ("rmse", "mae", "mbe", "sky_balanced_accuracy", "sky_macro_f1"):
        row[f"{key}_last"] = last.get(key)
        row[f"{key}_val"] = val.get(key)
    run_dir = OUT / name / "run"
    for tag, report in (("", "eval-test"), ("_last", "eval-test-last"), ("_val", "eval-val")):
        parquet = run_dir / report / "predictions.parquet"
        if parquet.exists():
            block = runner.score_by_sensor_block(pd.read_parquet(parquet), n_bootstrap=200)
            row[f"block_macro_f1{tag}"] = block["sky"]["macro_f1"]
            row[f"block_rmse{tag}"] = block["dhi"]["rmse"]
    print(
        f"{name:<18} {row.get('status')}  por bloco: F1 {row.get('block_macro_f1')} "
        f"(last {row.get('block_macro_f1_last')}, val {row.get('block_macro_f1_val')})  "
        f"RMSE {row.get('block_rmse')} (last {row.get('block_rmse_last')})"
    )
    print(" ", runner.archive(str(OUT / name), ARTIFACTS, config=config))
    row["arm"] = name.rsplit("_s", 1)[0]
    rows.append(row)
    return row


for seed in (42, 43, 44):
    arm(f"ceuS_s{seed}", seed, "arm S: o braco ceu local, nesta GPU — o portao de transferencia")

runner.summarise(rows)

## 6. O portão

Compara o arm S com a referência congelada (obrigatório) e com o `ceu` local (quando
preenchido). Sem amostra não há portão: menos de três sementes concluídas pára a execução.

In [ ]:
import numpy as np


def arm_rows(prefix):
    """Concluded rows of one arm."""
    return [r for r in rows if r.get("status") == "ok" and r.get("arm") == prefix]


gate = arm_rows("ceuS")
if len(gate) < 3:
    raise RuntimeError(f"{3 - len(gate)} semente(s) do arm S falharam — nenhum portao medido")
bal = np.array([r["sky_balanced_accuracy"] for r in gate], dtype=float)
block_f1 = np.array([r["block_macro_f1"] for r in gate], dtype=float)
rmse = np.array([r["rmse_last"] for r in gate], dtype=float)
if not (np.isfinite(bal).all() and np.isfinite(rmse).all() and np.isfinite(block_f1).all()):
    raise RuntimeError("metrica nao finita no arm S — diagnostique antes de seguir")
print(f"arm S: acuracia balanceada por frame {bal.mean():.3f} +- {bal.std(ddof=1):.3f}")
print(f"arm S: macro-F1 por bloco         {block_f1.mean():.3f} +- {block_f1.std(ddof=1):.3f}")
print(f"arm S: RMSE de DHI (last)         {rmse.mean():.2f} +- {rmse.std(ddof=1):.2f} W/m2")
if bal.mean() <= FROZEN_REFERENCE_BALANCED_ACCURACY:
    raise RuntimeError(
        f"fine-tuning nao superou a referencia congelada ({FROZEN_REFERENCE_BALANCED_ACCURACY}); "
        "os arms abaixo herdariam o defeito — pare aqui"
    )
local_bal = LOCAL_REFERENCE.get("sky_balanced_accuracy")
local_rmse = LOCAL_REFERENCE.get("dhi_rmse")
if local_bal is not None and abs(bal.mean() - local_bal) > 0.05:
    print(f"ATENCAO: {bal.mean():.3f} vs {local_bal:.3f} local — mais de 5 pontos; investigue")
if local_rmse is not None and abs(rmse.mean() - local_rmse) > 2.0:
    print(f"ATENCAO: {rmse.mean():.2f} vs {local_rmse:.2f} W/m2 local — mais de 2 W/m2; investigue")
print("portao aberto")

## 7. Arm U — a unidade do rótulo

`sensor_block`: a amostra são os frames que partilham a linha do CR5000 (ceil do carimbo local
a 5 min), média das codificações; `one_sample_per_block` treina e valida com o frame mais
próximo do centroide, então os forwards por época são os do arm S e os passos do otimizador
caem 5× — daí 60 épocas. É o único arm com mecanismo para passar do teto de 0,73 por frame.

In [ ]:
BLOCK_ALIGNMENT = {
    "strategy": "sensor_block",
    "window_minutes": 5.0,
    "max_frames": 5,
    "one_sample_per_block": True,
}

for seed in (42, 43, 44):
    arm(
        f"ceuU_s{seed}",
        seed,
        "arm U: uma amostra = os frames da mesma linha do datalogger",
        train={"epochs": 60, "early_stopping": {"patience": 60, "monitor": "val/loss"}},
        alignment=BLOCK_ALIGNMENT,
    )

runner.summarise(rows)

## 8. O vencedor é escolhido na validação, por bloco

Escolher no teste é o otimismo de seleção que a memória do projeto registra. `val` é 2× mais
difícil que o teste (35 % nublado), então os números abaixo são menores — e é isso que se compara.

In [ ]:
def mean_of(prefix, key):
    """Mean of one metric over the concluded seeds of an arm."""
    values = [r[key] for r in arm_rows(prefix) if r.get(key) is not None]
    return float(np.mean(values)) if values else float("nan")


def winner_of(*prefixes):
    """The arm with the best validation macro-F1 per block; ties go to the earlier one."""
    scored = [(mean_of(p, "block_macro_f1_val"), -i, p) for i, p in enumerate(prefixes)]
    return max(scored)[2]


DECISION_FLOOR = 0.03
WINNER_SU = winner_of("ceuS", "ceuU")
print(
    "S vs U por bloco na validacao:",
    {p: round(mean_of(p, "block_macro_f1_val"), 3) for p in ("ceuS", "ceuU")},
)
if mean_of("ceuU", "block_macro_f1_val") - mean_of("ceuS", "block_macro_f1_val") < DECISION_FLOOR:
    print(
        f"U nao passa do piso de decisao ({DECISION_FLOOR}) sobre S na validacao — registre como nulo"
    )
SETTINGS = {
    "ceuS": {"train": {}, "alignment": None},
    "ceuU": {
        "train": {"epochs": 60, "early_stopping": {"patience": 60, "monitor": "val/loss"}},
        "alignment": BLOCK_ALIGNMENT,
    },
}
print("vencedor S/U:", WINNER_SU)

## 9. Arm O — alvo ordinal e pesos por classe, sobre o vencedor

`class_weights` inverso-frequência do treino (0,648/1,028/1,845/0,478) e `ordinal_tau` 0,4
(rótulo suave de Díaz & Marathe 2019: 85 % na classe certa, 7 % em cada vizinha). Não move o
teto; ataca o recall das parciais e a CE de validação que explode por sobreconfiança.

In [ ]:
CEUORD_TARGETS = {
    **runner.CEU_TARGETS,
    "sky": {
        "enabled": True,
        "weight": 1.0,
        "class_weights": [0.648, 1.028, 1.845, 0.478],
        "ordinal_tau": 0.4,
    },
}
for seed in (42, 43, 44):
    arm(
        f"ceuO_s{seed}",
        seed,
        f"arm O: alvo ordinal + pesos por classe sobre {WINNER_SU}",
        targets=CEUORD_TARGETS,
        **SETTINGS[WINNER_SU],
    )
SETTINGS["ceuO"] = {**SETTINGS[WINNER_SU]}
WINNER_SUO = winner_of(WINNER_SU, "ceuO")
print("vencedor apos O:", WINNER_SUO)
runner.summarise(rows)

## 10. Arm B — capacidade, com portão

ViT-B/14, batch 48, só se o vencedor até aqui tiver passado do arm S por mais que o piso de
decisão na validação; caso contrário a capacidade só amplifica ruído de fronteira.

In [ ]:
RUN_B = (
    mean_of(WINNER_SUO, "block_macro_f1_val") - mean_of("ceuS", "block_macro_f1_val")
    >= DECISION_FLOOR
    or WINNER_SUO == "ceuS"
)
if RUN_B:
    base_targets = CEUORD_TARGETS if WINNER_SUO == "ceuO" else runner.CEU_TARGETS
    for seed in (42, 43):
        arm(
            f"ceuB_s{seed}",
            seed,
            f"arm B: ViT-B/14 sobre {WINNER_SUO}",
            model={"backbone": "dinov2_vitb14"},
            train={**SETTINGS[WINNER_SUO]["train"], "batch_size": 48},
            alignment=SETTINGS[WINNER_SUO]["alignment"],
            targets=base_targets,
        )
    SETTINGS["ceuB"] = {
        **SETTINGS[WINNER_SUO],
        "train": {**SETTINGS[WINNER_SUO]["train"], "batch_size": 48},
        "model": {"backbone": "dinov2_vitb14"},
    }
else:
    print("arm B pulado: nada passou do piso de decisao sobre S na validacao")
runner.summarise(rows)

## 11. Arm E — ensemble de cinco sementes do vencedor, escolhido na validação

Duas sementes extras do vencedor (validação, por bloco). O `ensemble_predictions` do runner dá a
difusa e o k\* pela média, a classe por **dois estimadores** (voto e Kt reconstruído) e a tabela
**por bloco** com IC bootstrap e a baseline do bloco anterior. Depois o heterogêneo com todos os
arms concluídos — o que mais ganhou na difusa localmente (16,6 W/m²).

In [ ]:
CANDIDATES = [p for p in ("ceuS", "ceuU", "ceuO", "ceuB") if arm_rows(p)]
BEST = winner_of(*CANDIDATES)
print("melhor arm pela validacao por bloco:", BEST)
extra = SETTINGS[BEST]
extra_model = extra.get("model", {})
extra_targets = (
    CEUORD_TARGETS
    if BEST == "ceuO" or (BEST == "ceuB" and WINNER_SUO == "ceuO")
    else runner.CEU_TARGETS
)
for seed in (45, 46):
    arm(
        f"{BEST}_s{seed}",
        seed,
        f"arm E: membro extra do ensemble de {BEST}",
        model=extra_model,
        train=extra["train"],
        alignment=extra["alignment"],
        targets=extra_targets,
    )

for split in ("val", "test"):
    report_dir = "eval-val" if split == "val" else "eval-test"
    members = sorted(Path(ARTIFACTS).glob(f"{BEST}_s*/{report_dir}/predictions.parquet"))
    if len(members) < 2:
        raise RuntimeError(f"{len(members)} membro(s) de {BEST} em {split} — sem ensemble")
    report = runner.ensemble_predictions(members, Path(ARTIFACTS) / f"ensemble_{BEST}_{split}")
    block = report["by_sensor_block"]
    print(
        f"[{split}] {BEST} x{len(members)}: por bloco macro-F1 {block['sky']['macro_f1']:.3f} IC95 {block['ci95']['sky_macro_f1']}"
        f" | bloco anterior {block['sky_persistence_previous_block']['macro_f1']:.3f} | DHI RMSE {block['dhi']['rmse']:.2f}"
    )
    for estimator, metrics in report.get("sky", {}).items():
        print(
            f"   por frame/{estimator}: bal_acc {metrics['balanced_accuracy']:.3f} macro-F1 {metrics['macro_f1']:.3f}"
        )

everyone = sorted(Path(ARTIFACTS).glob("ceu[SUOB]_s*/eval-test-last/predictions.parquet"))
if len(everyone) > 1:
    hetero = runner.ensemble_predictions(
        everyone,
        Path(ARTIFACTS) / "ensemble_heterogeneo_last",
        reference=sorted(Path(ARTIFACTS).glob("ceuS_s*/eval-test/predictions.parquet")),
    )
    print(
        f"heterogeneo (last, {len(everyone)} membros): DHI RMSE {hetero['dhi']['rmse']:.2f} MAE {hetero['dhi']['mae']:.2f} MBE {hetero['dhi']['mbe']:+.2f}"
        f" | por bloco macro-F1 {hetero['by_sensor_block']['sky']['macro_f1']:.3f}"
    )

## 12. Fechamento

Grava o índice da campanha, ordenado pelo macro-F1 **por bloco na validação** — o critério de
seleção — com a difusa ao lado para dizer quanto ela pagou pela classe.

In [ ]:
frame = runner.summarise(rows)
if "block_macro_f1_val" in frame.columns:
    frame = frame.sort_values("block_macro_f1_val", ascending=False, na_position="last")
frame.to_csv(f"{ARTIFACTS}/campanha_ceu.csv", index=False)
summary = {
    "hardware": HW,
    "n_runs": len(rows),
    "melhor_arm": BEST,
    "ensemble_test": report,
    "heterogeneo": hetero if len(everyone) > 1 else None,
}
with open(f"{ARTIFACTS}/campanha_ceu_resumo.json", "w") as handle:
    json.dump(summary, handle, indent=2, default=str)
print(frame.to_string())
print("artefatos em", ARTIFACTS)